# Motion

This notebook uses the Poetry-managed environment. All dependencies from `pyproject.toml` are available.


In [ ]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")
print(f"Python version: {sys.version}")

In [ ]:
from femtum_sdk.api.contracts.v1.hardware.health_pb2 import HealthRequest
from femtum_sdk.api import HardwareApi

# api_host = "10.20.200.10:5210"
api_host = "localhost:5210"

with HardwareApi(host=api_host) as hardware_api:
    print(hardware_api.health.GetHealth(HealthRequest()))


## Read Current Position

In [ ]:
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import AngleMeasureUnit, DistanceMeasureUnit
from femtum_sdk.api.contracts.v1.hardware.motion_pb2 import ReadBaseStageCurrentPositionRequest, ReadBaseStageCurrentPositionResponse

def print_position(response: ReadBaseStageCurrentPositionResponse):
    print(f"X = {response.X.Value} {DistanceMeasureUnit.Name(response.X.Unit)}")
    print(f"Y = {response.Y.Value} {DistanceMeasureUnit.Name(response.Y.Unit)}")
    print(f"Z = {response.Z.Value} {DistanceMeasureUnit.Name(response.Z.Unit)}")
    print(f"RX = {response.RX.Value} {AngleMeasureUnit.Name(response.RX.Unit)}")
    print(f"RY = {response.RY.Value} {AngleMeasureUnit.Name(response.RY.Unit)}")
    print(f"RZ = {response.RZ.Value} {AngleMeasureUnit.Name(response.RZ.Unit)}")

with HardwareApi(host=api_host) as hardware_api:
    response: ReadBaseStageCurrentPositionResponse = hardware_api.base_stage_motion.ReadCurrentPosition(ReadBaseStageCurrentPositionRequest())


    print_position(response)


## Relative Move

⚠️ **Warning:** Before running a move, ensure the path is clear to prevent collisions or equipment damage.

### All At Once

In [ ]:
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasure
from femtum_sdk.api.contracts.v1.hardware.motion_pb2 import AllAtOnceMove, MoveOrder, MoveToBaseStageRelativePositionRequest, MoveToBaseStageRelativePositionResponse


with HardwareApi(host=api_host) as hardware_api:
    initial_position: ReadBaseStageCurrentPositionResponse = hardware_api.base_stage_motion.ReadCurrentPosition(ReadBaseStageCurrentPositionRequest())

    after_move_position: MoveToBaseStageRelativePositionResponse = hardware_api.base_stage_motion.MoveToRelativePosition(MoveToBaseStageRelativePositionRequest(
        X=DistanceMeasure(
            Value=100,
            Unit=DistanceMeasureUnit.Microns
        ),
        Y=DistanceMeasure(
            Value=100, 
            Unit=DistanceMeasureUnit.Microns
        ),
        Order=MoveOrder(AllAtOnce=AllAtOnceMove())
    ))

    read_after_move_position: ReadBaseStageCurrentPositionResponse = hardware_api.base_stage_motion.ReadCurrentPosition(ReadBaseStageCurrentPositionRequest())

    print("Before move:")
    print_position(initial_position)
    print("After move:")
    print_position(read_after_move_position)
    tolerance = 0.5  # Microns; the stage won't land on the exact target
    assert abs(read_after_move_position.X.Value - (initial_position.X.Value + 100)) <= tolerance
    assert abs(read_after_move_position.Y.Value - (initial_position.Y.Value + 100)) <= tolerance
    assert abs(read_after_move_position.Z.Value - initial_position.Z.Value) <= tolerance

    assert abs(read_after_move_position.X.Value - (initial_position.X.Value + 100)) <= tolerance
    assert abs(read_after_move_position.Y.Value - (initial_position.Y.Value + 100)) <= tolerance
    assert abs(read_after_move_position.Z.Value - initial_position.Z.Value) <= tolerance


### Ordered

In [ ]:
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasure
from femtum_sdk.api.contracts.v1.dto.motion_pb2 import Axis
from femtum_sdk.api.contracts.v1.hardware.motion_pb2 import MoveOrder, MoveToBaseStageRelativePositionRequest, MoveToBaseStageRelativePositionResponse, OrderedMove


with HardwareApi(host=api_host) as hardware_api:
    initial_position: ReadBaseStageCurrentPositionResponse = hardware_api.base_stage_motion.ReadCurrentPosition(ReadBaseStageCurrentPositionRequest())

    move_a: MoveToBaseStageRelativePositionResponse = hardware_api.base_stage_motion.MoveToRelativePosition(MoveToBaseStageRelativePositionRequest(
        X=DistanceMeasure(
            Value=10000,
            Unit=DistanceMeasureUnit.Microns
        ),
        Y=DistanceMeasure(
            Value=10000, 
            Unit=DistanceMeasureUnit.Microns
        ),
        Z=DistanceMeasure(
            Value=10000,
            Unit=DistanceMeasureUnit.Microns
        ),
        Order=MoveOrder(Ordered=OrderedMove(Axis=[Axis.AXIS_Z, Axis.AXIS_Y, Axis.AXIS_X])) # Move Z first, then Y, then X
    ))
    print("Move A:")
    print_position(move_a)

    import time
    time.sleep(2.5) # Wait for human to see the move

    move_b: MoveToBaseStageRelativePositionResponse = hardware_api.base_stage_motion.MoveToRelativePosition(MoveToBaseStageRelativePositionRequest(
        X=DistanceMeasure(
            Value=-10000,
            Unit=DistanceMeasureUnit.Microns
        ),
        Y=DistanceMeasure(
            Value=-10000, 
            Unit=DistanceMeasureUnit.Microns
        ),
        Z=DistanceMeasure(
            Value=-10000,
            Unit=DistanceMeasureUnit.Microns
        ),
        Order=MoveOrder(Ordered=OrderedMove(Axis=[Axis.AXIS_X, Axis.AXIS_Y, Axis.AXIS_Z])) # Move X first, then Y, then Z
    ))

    print("Move B:")
    print_position(move_b)

    read_after_move_position: ReadBaseStageCurrentPositionResponse = hardware_api.base_stage_motion.ReadCurrentPosition(ReadBaseStageCurrentPositionRequest())

    tolerance = 0.5  # Microns; the stage won't land on the exact target
    assert abs(read_after_move_position.X.Value - move_b.X.Value) <= tolerance
    assert abs(read_after_move_position.Y.Value - move_b.Y.Value) <= tolerance
    assert abs(read_after_move_position.Z.Value - move_b.Z.Value) <= tolerance


## Absolute Move

⚠️ **Warning:** Before running a move, ensure the path is clear to prevent collisions or equipment damage.

### All at Once

In [ ]:
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasure
from femtum_sdk.api.contracts.v1.hardware.motion_pb2 import MoveToBaseStageAbsolutePositionRequest, MoveToBaseStageAbsolutePositionResponse


with HardwareApi(host=api_host) as hardware_api:
    initial_position: ReadBaseStageCurrentPositionResponse = hardware_api.base_stage_motion.ReadCurrentPosition(ReadBaseStageCurrentPositionRequest())

    move_a_response: MoveToBaseStageAbsolutePositionResponse = hardware_api.base_stage_motion.MoveToAbsolutePosition(MoveToBaseStageAbsolutePositionRequest(
        X=DistanceMeasure(
            Value=25200.0,
            Unit=DistanceMeasureUnit.Microns
        ),
        Y=DistanceMeasure(
            Value=6574.0, 
            Unit=DistanceMeasureUnit.Microns
        ),
    ))

    move_b_response: MoveToBaseStageAbsolutePositionResponse = hardware_api.base_stage_motion.MoveToAbsolutePosition(MoveToBaseStageAbsolutePositionRequest(
        X=DistanceMeasure(
            Value=25300.0,
            Unit=DistanceMeasureUnit.Microns
        ),
        Y=DistanceMeasure(
            Value=6580.0, 
            Unit=DistanceMeasureUnit.Microns
        ),
    ))

    read_after_move_position: ReadBaseStageCurrentPositionResponse = hardware_api.base_stage_motion.ReadCurrentPosition(ReadBaseStageCurrentPositionRequest())

    print("Before move:")
    print_position(initial_position)
    print("After move:")
    print_position(read_after_move_position)
    tolerance = 0.5  # Microns; the stage won't land on the exact target
    assert abs(read_after_move_position.X.Value - move_b_response.X.Value) <= tolerance
    assert abs(read_after_move_position.Y.Value - move_b_response.Y.Value) <= tolerance


### Ordered

In [ ]:
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasure
from femtum_sdk.api.contracts.v1.hardware.motion_pb2 import MoveToBaseStageAbsolutePositionRequest, MoveToBaseStageAbsolutePositionResponse


with HardwareApi(host=api_host) as hardware_api:
    initial_position: ReadBaseStageCurrentPositionResponse = hardware_api.base_stage_motion.ReadCurrentPosition(ReadBaseStageCurrentPositionRequest())

    move_a_response: MoveToBaseStageAbsolutePositionResponse = hardware_api.base_stage_motion.MoveToAbsolutePosition(MoveToBaseStageAbsolutePositionRequest(
        X=DistanceMeasure(
            Value=35200.0,
            Unit=DistanceMeasureUnit.Microns
        ),
        Y=DistanceMeasure(
            Value=8574.0, 
            Unit=DistanceMeasureUnit.Microns
        ),
        Order=MoveOrder(Ordered=OrderedMove(Axis=[Axis.AXIS_Y, Axis.AXIS_X]))
    ))

    import time
    time.sleep(2.5) # Wait for human to see the move

    move_b_response: MoveToBaseStageAbsolutePositionResponse = hardware_api.base_stage_motion.MoveToAbsolutePosition(MoveToBaseStageAbsolutePositionRequest(
        X=DistanceMeasure(
            Value=25300.0,
            Unit=DistanceMeasureUnit.Microns
        ),
        Y=DistanceMeasure(
            Value=6580.0, 
            Unit=DistanceMeasureUnit.Microns
        ),
        Order=MoveOrder(Ordered=OrderedMove(Axis=[Axis.AXIS_X, Axis.AXIS_Y]))
    ))

    read_after_move_position: ReadBaseStageCurrentPositionResponse = hardware_api.base_stage_motion.ReadCurrentPosition(ReadBaseStageCurrentPositionRequest())

    print("Before move:")
    print_position(initial_position)
    print("After move:")
    print_position(read_after_move_position)

